# Zero Coupon Curve Bootstrapping

A **zero-coupon curve** maps each maturity to the discount factor that makes one unit of cash received at that date worth its present value today. Bootstrapping recovers those discount factors sequentially from market instrument prices, solving one pillar at a time in maturity order.

This notebook builds two curves that together define the **USD multi-curve framework** as of 2 January 2024:

| Curve | Instruments | Purpose |
|---|---|---|
| **OIS discount curve** | SOFR OIS swaps (1M–10Y) | Discounting cash flows (CSA-collateralised trades) |
| **Projection curve** | Deposits + SOFR futures + SOFR swaps | Projecting floating rate cash flows |

The `ZeroCurveBootstrapper` accepts four instrument types:
- **`DepositQuote`** — money-market deposits; directly constrain the front end.
- **`FuturesQuote`** — IMM-dated exchange-traded contracts; bridge front end to mid curve. A user-supplied **convexity adjustment** corrects for the daily mark-to-market bias relative to a forward rate agreement.
- **`OISQuote`** — overnight index swaps; build the risk-free discount curve using the **continuous approximation** for the floating leg.
- **`SwapQuote`** — plain-vanilla interest rate swaps; build the projection curve in a **multi-curve** setup, discounting on an external OIS curve.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == 'examples' else Path.cwd()
sys.path.insert(0, str(root))

from database import set_db_path
set_db_path(str(root / 'examples' / 'demo.db'))

In [ ]:
import sqlite3
from database import get_db_path
from scripts.initialise import init_db, _seed_holidays

init_db()
with sqlite3.connect(get_db_path()) as conn:
    count = conn.execute("SELECT COUNT(*) FROM holidays").fetchone()[0]
    if count == 0:
        _seed_holidays(conn)
        print("Seeded demo.db with holiday data.")
    else:
        print(f"demo.db already seeded ({count} holidays). Skipping.")

In [ ]:
from datetime import date
import warnings
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from market_conventions import BusinessDayConvention, DayCountConvention, StubType
from market_structures import (
    ZeroCurveBootstrapper,
    DepositQuote,
    FuturesQuote,
    OISQuote,
    SwapQuote,
    MaturityReference,
    QuoteHierarchy,
)
from schedules.date_utils import imm_date, add_tenor
from schedules.calendars import CalendarType, HolidayCalendar
from schedules.schedule import Frequency

# ── shared conventions ──────────────────────────────────────────────────────
REF    = date(2024, 1, 2)          # pricing date
USD    = CalendarType.USD
MF     = BusinessDayConvention.MODIFIED_FOLLOWING
ACT360 = DayCountConvention.ACT_360
ACT365 = DayCountConvention.ACT_365_FIXED

print(f"Reference date: {REF}")

## Part I — OIS Discount Curve

The SOFR OIS curve is the standard USD discount curve for collateralised derivatives. Each OIS swap exchanges the fixed rate for the daily-compounded SOFR overnight index. The bootstrapper uses a **continuous approximation** for the floating leg:

$$\text{Floating PV} \approx P(T_0) - P(T_n)$$

where $P(\cdot)$ denotes OIS discount factors (the same curve being built — OIS is self-discounting). This is exact in the continuous limit and negligibly different from exact daily compounding for typical accrual periods.

We quote nine SOFR OIS tenors from 1M to 10Y, reflecting a US rate environment with the Fed funds target at 5.25–5.50% and the market pricing approximately 150 bps of cuts over the following two years.

In [ ]:
ois_quotes = [
    OISQuote(rate=0.0532, tenor="1M",  spot_lag=2, frequency=Frequency.ANNUAL,
             calendar=USD, business_day_convention=MF, day_count_convention=ACT360),
    OISQuote(rate=0.0528, tenor="3M",  spot_lag=2, frequency=Frequency.ANNUAL,
             calendar=USD, business_day_convention=MF, day_count_convention=ACT360),
    OISQuote(rate=0.0517, tenor="6M",  spot_lag=2, frequency=Frequency.ANNUAL,
             calendar=USD, business_day_convention=MF, day_count_convention=ACT360),
    OISQuote(rate=0.0498, tenor="1Y",  spot_lag=2, frequency=Frequency.ANNUAL,
             calendar=USD, business_day_convention=MF, day_count_convention=ACT360),
    OISQuote(rate=0.0455, tenor="2Y",  spot_lag=2, frequency=Frequency.ANNUAL,
             calendar=USD, business_day_convention=MF, day_count_convention=ACT360),
    OISQuote(rate=0.0435, tenor="3Y",  spot_lag=2, frequency=Frequency.ANNUAL,
             calendar=USD, business_day_convention=MF, day_count_convention=ACT360),
    OISQuote(rate=0.0410, tenor="5Y",  spot_lag=2, frequency=Frequency.ANNUAL,
             calendar=USD, business_day_convention=MF, day_count_convention=ACT360),
    OISQuote(rate=0.0402, tenor="7Y",  spot_lag=2, frequency=Frequency.ANNUAL,
             calendar=USD, business_day_convention=MF, day_count_convention=ACT360),
    OISQuote(rate=0.0395, tenor="10Y", spot_lag=2, frequency=Frequency.ANNUAL,
             calendar=USD, business_day_convention=MF, day_count_convention=ACT360),
]

ois_bootstrapper = ZeroCurveBootstrapper(
    reference_date=REF,
    quotes=ois_quotes,
    day_count_convention=ACT365,
    tolerance=1e-10,
    max_iterations=50,
)

ois_curve = ois_bootstrapper.bootstrap()
print(f"OIS curve bootstrapped: {len(ois_curve._pillar_dates)} pillars")

In [ ]:
print(ois_curve.summary())

## Part II — Projection Curve (Multi-Curve)

In the multi-curve framework, a separate **projection curve** determines expected floating-rate cash flows. It is built from the same types of market instruments as the OIS curve but is discounted on the **OIS curve** — reflecting that collateralised swaps earn the overnight rate on posted margin.

We use three instrument types in sequence:

1. **Money-market deposits** (3M, 6M) — pin the front end. Settlement is T+2; day count ACT/360.
2. **SOFR futures** (H25, M25, U25, Z25) — pin the 1–2Y segment. IMM-dated quarterly contracts with 3M tenors. Small convexity adjustments convert futures rates to forward rates.
3. **Vanilla IRS** (2Y–10Y) — pin the long end. Fixed leg: semi-annual ACT/360. Floating leg: quarterly ACT/360. Discounted on the OIS curve.

In [ ]:
# ── 1. Money-market deposits ────────────────────────────────────────────────
deposit_quotes = [
    DepositQuote(rate=0.0529, tenor="3M", spot_lag=2,
                 calendar=USD, business_day_convention=MF,
                 day_count_convention=ACT360),
    DepositQuote(rate=0.0519, tenor="6M", spot_lag=2,
                 calendar=USD, business_day_convention=MF,
                 day_count_convention=ACT360),
]

# ── 2. SOFR futures (IMM-dated, 3M tenor) ───────────────────────────────────
# Convexity adjustment (bps → decimal) corrects the futures rate for the
# daily mark-to-market effect absent in a forward rate agreement.
futures_quotes = [
    FuturesQuote(price=95.20, imm_code="H25", tenor="3M",
                 calendar=USD, business_day_convention=MF,
                 day_count_convention=ACT360, convexity_adjustment=0.0001),
    FuturesQuote(price=95.50, imm_code="M25", tenor="3M",
                 calendar=USD, business_day_convention=MF,
                 day_count_convention=ACT360, convexity_adjustment=0.0002),
    FuturesQuote(price=95.75, imm_code="U25", tenor="3M",
                 calendar=USD, business_day_convention=MF,
                 day_count_convention=ACT360, convexity_adjustment=0.0003),
    FuturesQuote(price=95.95, imm_code="Z25", tenor="3M",
                 calendar=USD, business_day_convention=MF,
                 day_count_convention=ACT360, convexity_adjustment=0.0005),
]

# Show IMM dates and adjusted rates
print("Futures contract details:")
print(f"{'Code':<6} {'IMM Start':<14} {'Contract End':<14} {'Price':>7} {'Adj Rate':>10}")
print("-" * 55)
cal = HolidayCalendar(USD)
for q in futures_quotes:
    start = imm_date(q.imm_code)
    end   = q.maturity_date(REF)
    print(f"{q.imm_code:<6} {str(start):<14} {str(end):<14} {q.price:>7.2f} {q.initial_guess():>9.4%}")

In [ ]:
# ── 3. USD SOFR vanilla swaps (multi-curve, discounted on OIS) ──────────────
swap_quotes = [
    SwapQuote(rate=0.0458, tenor="2Y",  spot_lag=2,
              fixed_frequency=Frequency.SEMI_ANNUAL, fixed_day_count=ACT360,
              floating_frequency=Frequency.QUARTERLY, floating_day_count=ACT360,
              calendar=USD, business_day_convention=MF,
              discount_curve=ois_curve),
    SwapQuote(rate=0.0438, tenor="3Y",  spot_lag=2,
              fixed_frequency=Frequency.SEMI_ANNUAL, fixed_day_count=ACT360,
              floating_frequency=Frequency.QUARTERLY, floating_day_count=ACT360,
              calendar=USD, business_day_convention=MF,
              discount_curve=ois_curve),
    SwapQuote(rate=0.0414, tenor="5Y",  spot_lag=2,
              fixed_frequency=Frequency.SEMI_ANNUAL, fixed_day_count=ACT360,
              floating_frequency=Frequency.QUARTERLY, floating_day_count=ACT360,
              calendar=USD, business_day_convention=MF,
              discount_curve=ois_curve),
    SwapQuote(rate=0.0398, tenor="10Y", spot_lag=2,
              fixed_frequency=Frequency.SEMI_ANNUAL, fixed_day_count=ACT360,
              floating_frequency=Frequency.QUARTERLY, floating_day_count=ACT360,
              calendar=USD, business_day_convention=MF,
              discount_curve=ois_curve),
]

# ── Bootstrap projection curve ──────────────────────────────────────────────
all_proj_quotes = deposit_quotes + futures_quotes + swap_quotes

proj_bootstrapper = ZeroCurveBootstrapper(
    reference_date=REF,
    quotes=all_proj_quotes,
    day_count_convention=ACT365,
    tolerance=1e-10,
    max_iterations=50,
)

proj_curve = proj_bootstrapper.bootstrap()
print(proj_curve.summary())

## Round-Trip Verification

A correctly bootstrapped curve reproduces its input instrument prices exactly. We re-price each instrument on the bootstrapped curve and report the NPV residual. All residuals must be negligible (below the calibration tolerance of 1e-10).

In [ ]:
tenors = ["1M", "3M", "6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y"]
labels = ["3M dep", "6M dep", "H25 fut", "M25 fut", "U25 fut", "Z25 fut",
          "2Y swap", "3Y swap", "5Y swap", "10Y swap"]

print("OIS curve — round-trip check")
print(f"{'Tenor':<8} {'NPV residual':>16}  {'Pass?':>6}")
print("-" * 36)
for label, q in zip(tenors, ois_quotes):
    resid = q.npv(REF, ois_curve)
    ok    = abs(resid) < 1e-8
    print(f"{label:<8} {resid:>16.2e}  {'✓' if ok else '✗':>6}")

print()
print("Projection curve — round-trip check")
print(f"{'Instrument':<12} {'NPV residual':>16}  {'Pass?':>6}")
print("-" * 40)
for label, q in zip(labels, all_proj_quotes):
    resid = q.npv(REF, proj_curve)
    ok    = abs(resid) < 1e-8
    print(f"{label:<12} {resid:>16.2e}  {'✓' if ok else '✗':>6}")

## Resolving Duplicate Maturities: QuoteHierarchy

When two instruments share the same bootstrapping pillar date, `ZeroCurveBootstrapper` resolves the conflict automatically using `QuoteHierarchy`. The lower-priority instrument is silently discarded with a `UserWarning`; bootstrapping continues on the winner.

Fixed priority (lower rank wins):

| Rank | Type |
|---|---|
| 1 | `DepositQuote` |
| 2 | `OISQuote` |
| 3 | `SwapQuote` |
| 4 | `FuturesQuote` |

The ordering reflects standard market-data precedence: a money-market deposit is the most direct observation of the short-term rate and should always take precedence over a swap or futures quote at the same maturity. Every new `MarketQuote` subclass must be registered in `QuoteHierarchy._RANK`.

In [ ]:
# A 3M deposit and a 3M OIS landing on the same maturity date
dep_3m = DepositQuote(rate=0.0529, tenor="3M", spot_lag=2,
                      calendar=USD, business_day_convention=MF,
                      day_count_convention=ACT360)
ois_3m = OISQuote(rate=0.0532, tenor="3M", spot_lag=2,
                  frequency=Frequency.ANNUAL, calendar=USD,
                  business_day_convention=MF, day_count_convention=ACT365)

print(f"Deposit maturity: {dep_3m.maturity_date(REF)}")
print(f"OIS maturity:     {ois_3m.maturity_date(REF)}")
print()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    curve_clash = ZeroCurveBootstrapper(
        reference_date=REF, quotes=[dep_3m, ois_3m],
        day_count_convention=ACT365,
    ).bootstrap()

for w in caught:
    print(f"UserWarning: {w.message}")

print(f"\nPillars after resolution: {len(curve_clash._pillar_dates)}")
print(f"Deposit NPV (winner):  {dep_3m.npv(REF, curve_clash):.2e}  ✓")
print(f"OIS NPV    (discarded): {ois_3m.npv(REF, curve_clash):.2e}")

## Curve Visualisation

### Zero Rate Curves

Both curves show the familiar inversion seen in early 2024 — the very short end anchored by the policy rate, the intermediate segment pricing significant cuts, and the long end settling at a lower terminal rate. The projection curve sits slightly above the OIS curve, reflecting the **SOFR basis**: the credit/liquidity premium of 3M SOFR fixings over the overnight rate.

In [ ]:
# Sample both curves monthly out to 10 years
from datetime import timedelta

def month_dates(ref, n_months):
    """Generate approximately monthly dates from ref."""
    dates = []
    for m in range(1, n_months + 1):
        month = (ref.month - 1 + m) % 12 + 1
        year  = ref.year + (ref.month - 1 + m) // 12
        import calendar as _cal
        day = min(ref.day, _cal.monthrange(year, month)[1])
        dates.append(date(year, month, day))
    return dates

plot_dates = month_dates(REF, 120)  # monthly, 10 years
t_axis     = [(d - REF).days / 365.25 for d in plot_dates]

ois_zero  = [ois_curve.zero_rate(d) * 100  for d in plot_dates]
proj_zero = [proj_curve.zero_rate(d) * 100 for d in plot_dates]

ois_pillars  = [(d - REF).days / 365.25 for d in ois_curve._pillar_dates]
proj_pillars = [(d - REF).days / 365.25 for d in proj_curve._pillar_dates]

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(t_axis, ois_zero,  color='steelblue',  linewidth=2, label='OIS (SOFR) discount curve')
ax.plot(t_axis, proj_zero, color='tomato',      linewidth=2, label='Projection curve (3M SOFR swaps)')
ax.scatter(ois_pillars,  [ois_curve.zero_rate(d)*100  for d in ois_curve._pillar_dates],
           color='steelblue', s=40, zorder=5)
ax.scatter(proj_pillars, [proj_curve.zero_rate(d)*100 for d in proj_curve._pillar_dates],
           color='tomato',     s=40, zorder=5)
ax.set_xlabel('Years')
ax.set_ylabel('Zero rate (%)')
ax.set_title('Zero Coupon Curves — USD, 2 January 2024')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Discount Factors

Discount factors from the OIS curve are the market's present value of a risk-free cash flow at each date. They reflect the cumulative effect of the time value of money and encode all information needed for CSA-discounting of derivative cash flows.

In [ ]:
ois_dfs  = [ois_curve.discount_factor(d)  for d in plot_dates]
proj_dfs = [proj_curve.discount_factor(d) for d in plot_dates]

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t_axis, ois_dfs,  color='steelblue', linewidth=2, label='OIS discount factors')
ax.plot(t_axis, proj_dfs, color='tomato',     linewidth=2, label='Projection curve discount factors')
ax.set_xlabel('Years')
ax.set_ylabel('Discount factor')
ax.set_title('Discount Factors — USD, 2 January 2024')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3-Month Forward Rate Curve

Forward rates are the market-implied rates for future lending periods, extracted directly from the discount factor curve. The 3M forward rate curve from the projection curve is what drives the floating leg cash flows on a quarterly SOFR swap.

The forward curve typically exhibits more curvature than the zero curve and is the key input for any floating rate instrument valuation.

In [ ]:
# 3M forward rates from both curves — rolling quarterly window
from datetime import timedelta
import calendar as _cal

def add_months(d, n):
    total = d.year * 12 + (d.month - 1) + n
    y, m = divmod(total, 12)
    return date(y, m + 1, min(d.day, _cal.monthrange(y, m + 1)[1]))

# Sample forward rates at quarterly intervals
fwd_starts  = [add_months(REF, m)     for m in range(0, 117, 3)]
fwd_ends    = [add_months(REF, m + 3) for m in range(0, 117, 3)]
fwd_t_axis  = [(s - REF).days / 365.25 for s in fwd_starts]

ois_fwds  = [ois_curve.forward_rate(s, e)  * 100 for s, e in zip(fwd_starts, fwd_ends)]
proj_fwds = [proj_curve.forward_rate(s, e) * 100 for s, e in zip(fwd_starts, fwd_ends)]

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.step(fwd_t_axis, ois_fwds,  where='post', color='steelblue', linewidth=1.8,
        label='OIS 3M forward rate')
ax.step(fwd_t_axis, proj_fwds, where='post', color='tomato',     linewidth=1.8,
        label='Projection 3M forward rate')
ax.set_xlabel('Years')
ax.set_ylabel('3M forward rate (%)')
ax.set_title('3-Month Forward Rate Curves — USD, 2 January 2024')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### OIS–Projection Basis

The spread between the projection and OIS zero rates — the **SOFR basis** — quantifies the premium priced into 3M SOFR swaps over the overnight rate. This basis arises from liquidity and credit considerations and is explicitly priced in the multi-curve framework.

In [ ]:
basis_bps = [(p - o) * 100 for p, o in zip(proj_zero, ois_zero)]

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.fill_between(t_axis, basis_bps, alpha=0.35, color='mediumpurple')
ax.plot(t_axis, basis_bps, color='mediumpurple', linewidth=1.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Years')
ax.set_ylabel('Basis (bps)')
ax.set_title('Projection − OIS Basis (bps) — SOFR Swap vs. OIS Zero Rate')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Querying the Curves

Both curves expose three core queries used in derivatives pricing:

| Method | Usage |
|---|---|
| `discount_factor(d)` | PV of 1 unit received at date `d` (OIS curve for CSA discounting) |
| `zero_rate(d)` | Continuously compounded zero rate to date `d` |
| `forward_rate(start, end)` | Implied forward rate for the period `[start, end]` (projection curve for floating legs) |

In [ ]:
# Example: price a fixed vs. floating cash flow exchange at 2Y
spot      = add_months(REF, 2)        # T+2 spot
maturity  = add_months(REF, 24)       # 2Y maturity
notional  = 10_000_000

df_mat    = ois_curve.discount_factor(maturity)
zero_2y   = ois_curve.zero_rate(maturity)
fwd_1y2y  = proj_curve.forward_rate(add_months(REF, 12), add_months(REF, 24))

print(f"2Y OIS discount factor:       {df_mat:.6f}")
print(f"2Y OIS zero rate (cont.):     {zero_2y:.4%}")
print(f"1Y–2Y 3M SOFR forward rate:   {fwd_1y2y:.4%}")
print()
print(f"PV of $10M received in 2Y:    ${notional * df_mat:>12,.0f}")

## Payment Lag

In standard USD swap conventions payment is made on the **same business day as the period end** (payment lag = 0). Some instruments — notably cross-currency swaps and certain structured products — use a **T+2 payment lag**: the coupon is paid two business days after the accrual period closes.

The `payment_lag` parameter on `OISQuote` and `SwapQuote` (passed through to `Schedule`) controls this offset. With a non-zero lag, `pay_date` is advanced by that many business days beyond the BDC-adjusted period end:

- The **fixed leg** discounts each coupon to its `pay_date`.
- The **floating leg** forward rate is still derived from the accrual window `[accrual_start, accrual_end]`, but the resulting coupon is discounted to `pay_date`.

Bootstrapping still converges cleanly — the Newton-Raphson objective uses the lagged pay dates throughout, so the calibrated zero rates are self-consistent with the payment convention.

In [ ]:
from datetime import timedelta
from schedules.schedule import Schedule

# Illustrate with the 3Y swap — its maturity falls on a Monday, giving 6 clean semi-annual periods
spot_date = date(2024, 1, 4)                    # T+2 from REF in USD calendar
mat_date  = swap_quotes[1].maturity_date(REF)   # 3Y MF-adjusted maturity

fixed_sched_t0 = Schedule(
    effective_date=spot_date, termination_date=mat_date,
    frequency=Frequency.SEMI_ANNUAL, day_count_convention=ACT360,
    business_day_convention=MF, calendar=USD, payment_lag=0,
).generate()

fixed_sched_t2 = Schedule(
    effective_date=spot_date, termination_date=mat_date,
    frequency=Frequency.SEMI_ANNUAL, day_count_convention=ACT360,
    business_day_convention=MF, calendar=USD, payment_lag=2,
).generate()

print(f"3Y fixed-leg schedule  (spot={spot_date}  mat={mat_date})  lag=0 vs. lag=2")
print(f"{'#':<3} {'Accrual end':<14} {'Pay (lag=0)':<14} {'Pay (lag=2)':<14}  Shift")
print("-" * 62)
for i, (p0, p2) in enumerate(zip(fixed_sched_t0, fixed_sched_t2), 1):
    shift = (p2.pay_date - p0.pay_date).days
    print(f"{i:<3} {str(p0.accrual_end):<14} {str(p0.pay_date):<14} {str(p2.pay_date):<14}  +{shift} cal days")

print()

# Bootstrap projection curve with T+2 payment lag on all swap instruments
swap_quotes_t2 = [
    SwapQuote(
        rate=q.rate, tenor=q.tenor, spot_lag=q.spot_lag,
        fixed_frequency=q.fixed_frequency, fixed_day_count=q.fixed_day_count,
        floating_frequency=q.floating_frequency, floating_day_count=q.floating_day_count,
        calendar=q.calendar, business_day_convention=q.bdc,
        discount_curve=ois_curve, payment_lag=2,
    )
    for q in swap_quotes
]

proj_t2_quotes = deposit_quotes + futures_quotes + swap_quotes_t2
proj_t2_curve  = ZeroCurveBootstrapper(
    reference_date=REF, quotes=proj_t2_quotes,
    day_count_convention=ACT365, tolerance=1e-10,
).bootstrap()

# Round-trip check
print("Round-trip — projection curve with payment_lag=2 on swaps:")
print(f"{'Instrument':<12} {'NPV residual':>16}  {'Pass?':>6}")
print("-" * 40)
labels_all = ["3M dep", "6M dep", "H25 fut", "M25 fut", "U25 fut", "Z25 fut",
              "2Y swap", "3Y swap", "5Y swap", "10Y swap"]
for lbl, q in zip(labels_all, proj_t2_quotes):
    resid = q.npv(REF, proj_t2_curve)
    print(f"{lbl:<12} {resid:>16.2e}  {'✓' if abs(resid) < 1e-8 else '✗':>6}")

# Zero-rate shift vs. lag=0 curve — small but non-trivial
print()
print("Zero rate shift lag=0 → lag=2 at swap pillars:")
for q0 in swap_quotes:
    mat = q0.maturity_date(REF)
    delta_bps = (proj_t2_curve.zero_rate(mat) - proj_curve.zero_rate(mat)) * 10_000
    print(f"  {q0.tenor:<5}  Δ = {delta_bps:+.4f} bps")

### Maturity Reference: Placing the Pillar at the Payment Date

By default the bootstrapping pillar is placed at the **accrual period end** (`MaturityReference.ACCRUAL_END`). Setting `maturity_reference=MaturityReference.PAYMENT_DATE` shifts the pillar itself forward by `payment_lag` business days, aligning it with the last actual cash flow date.

This distinction matters when the payment lag is non-trivial. With `ACCRUAL_END` the bootstrapper pins the zero rate at the accrual end and interpolates the discount factor to the payment date; with `PAYMENT_DATE` it directly pins the discount factor at the point of the cash flow, which is the more natural anchor for instruments where the economic exposure settles at payment.

In [ ]:
from schedules.calendars import HolidayCalendar

ois_ae = OISQuote(rate=0.0435, tenor="3Y", spot_lag=2, frequency=Frequency.ANNUAL,
                  calendar=USD, business_day_convention=MF, day_count_convention=ACT360,
                  payment_lag=2, maturity_reference=MaturityReference.ACCRUAL_END)

ois_pd = OISQuote(rate=0.0435, tenor="3Y", spot_lag=2, frequency=Frequency.ANNUAL,
                  calendar=USD, business_day_convention=MF, day_count_convention=ACT360,
                  payment_lag=2, maturity_reference=MaturityReference.PAYMENT_DATE)

cal = HolidayCalendar(USD)
ae_mat = ois_ae.maturity_date(REF)
pd_mat = ois_pd.maturity_date(REF)

print(f"ACCRUAL_END  pillar: {ae_mat}")
print(f"PAYMENT_DATE pillar: {pd_mat}  (+{(pd_mat - ae_mat).days} calendar days)")
print()

curve_ae = ZeroCurveBootstrapper(reference_date=REF, quotes=[ois_ae],
                                  day_count_convention=ACT365).bootstrap()
curve_pd = ZeroCurveBootstrapper(reference_date=REF, quotes=[ois_pd],
                                  day_count_convention=ACT365).bootstrap()

print(f"ACCRUAL_END  round-trip NPV: {ois_ae.npv(REF, curve_ae):.2e}  ✓")
print(f"PAYMENT_DATE round-trip NPV: {ois_pd.npv(REF, curve_pd):.2e}  ✓")
print()

# The zero rates at the respective pillar dates differ slightly
zr_ae = curve_ae.zero_rate(ae_mat)
zr_pd = curve_pd.zero_rate(pd_mat)
print(f"Zero rate at ACCRUAL_END  pillar: {zr_ae:.4%}")
print(f"Zero rate at PAYMENT_DATE pillar: {zr_pd:.4%}  (same instrument, different pillar placement)")

## Convexity Adjustment Sensitivity

The convexity adjustment corrects for the fact that futures P&L is settled daily (mark-to-market), whereas a forward rate agreement is settled at maturity. The adjustment is typically computed from an interest rate model (e.g. Hull-White) and grows with both volatility and time to expiry. Here we illustrate its effect on the bootstrapped zero rate at the H25 pillar.

In [ ]:
# Sweep the H25 convexity adjustment from 0 to 10 bps
dep_only = deposit_quotes + futures_quotes[1:] + swap_quotes  # H25 replaced below
ca_range = [i * 0.0001 for i in range(0, 11)]  # 0 to 10 bps
h25_zero_rates = []

h25_mat = futures_quotes[0].maturity_date(REF)

for ca in ca_range:
    h25_variant = FuturesQuote(
        price=95.20, imm_code="H25", tenor="3M",
        calendar=USD, business_day_convention=MF,
        day_count_convention=ACT360, convexity_adjustment=ca,
    )
    trial_quotes = deposit_quotes + [h25_variant] + futures_quotes[1:] + swap_quotes
    trial_curve  = ZeroCurveBootstrapper(
        reference_date=REF, quotes=trial_quotes,
        day_count_convention=ACT365,
    ).bootstrap()
    h25_zero_rates.append(trial_curve.zero_rate(h25_mat) * 100)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot([c * 10000 for c in ca_range], h25_zero_rates, color='steelblue', linewidth=2, marker='o', ms=5)
ax.set_xlabel('H25 convexity adjustment (bps)')
ax.set_ylabel('Bootstrapped zero rate at H25 maturity (%)')
ax.set_title('Effect of Convexity Adjustment on Bootstrapped Zero Rate (H25 pillar)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

delta_bps = (h25_zero_rates[-1] - h25_zero_rates[0]) * 100
print(f"Zero rate shift per bp of convexity adjustment: {delta_bps / 10:.2f} bps")